In [ ]:
import numpy as np
import scipy as sp
import pandas as pd
import corner
import itertools
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib
import matplotlib.cm as cm
from matplotlib.colors import LogNorm
from itertools import combinations_with_replacement

In [ ]:
from matplotlib import rc
rc('text', usetex=True)
rc('font', family='serif')
rc('mathtext', default='sf')
rc("lines", markeredgewidth=1)
rc("lines", linewidth=1)
rc('axes', titlesize=24) #24
rc('axes', labelsize=24) #24
rc("axes", linewidth=1) #2)
rc('axes', labelpad=20) #24)
rc('xtick', labelsize=16)
rc('ytick', labelsize=16)
rc('legend', fontsize=16)
rc('ytick', right='True',direction= 'in')
rc('xtick', top='True',direction= 'in')
rc('xtick.major', pad=15) #8)
rc('ytick.major', pad=15) #8)
rc('xtick.major', size=12) #8)
rc('ytick.major', size=12) #8)
rc('xtick.minor', size=7) #8)
rc('ytick.minor', size=7) #8)
rc('figure', titlesize=27) #24)

### Initialization

In [ ]:
df = pd.read_csv('../src/data/RIT_Parameters_non-spinning-equal-mass.csv')
df.set_index('ID', inplace=True)
masks = df['catalog'] == 'RIT'
rit_ids = df.index
chi = df.loc[rit_ids, 'chieff'].values
nu = df.loc[rit_ids, 'nu'].values
ecc = df.loc[rit_ids, 'ecc'].values
emrg = df.loc[rit_ids, 'Heff_til'].values
bmrg = df.loc[rit_ids, 'b_massless_EOB'].values
jmrg = df.loc[rit_ids, 'Jmrg_til'].values

In [ ]:
c2A, c3A, c2p, c3p, c4p = np.zeros(len(rit_ids)), np.zeros(len(rit_ids)), np.zeros(len(rit_ids)), np.zeros(len(rit_ids)), np.zeros(len(rit_ids))
c2A_err, c3A_err, c2p_err, c3p_err, c4p_err = np.zeros(len(rit_ids)), np.zeros(len(rit_ids)), np.zeros(len(rit_ids)), np.zeros(len(rit_ids)), np.zeros(len(rit_ids))
full_data = []
for i, rit_id in enumerate(rit_ids):
    try:
        try:
            log_L_1 = np.genfromtxt(f'../src/output/nc_fits_rit_non-spinning/RIT_{rit_id}_1/Algorithm/Evidence.txt')[1]
        except:
            log_L_1 = -np.inf
        try:
            log_L_2 = np.genfromtxt(f'../src/output/nc_fits_rit_non-spinning/RIT_{rit_id}_2/Algorithm/Evidence.txt')[1]
        except:
            log_L_2 = -np.inf
        try:
            log_L_3 = np.genfromtxt(f'../src/output/nc_fits_rit_non-spinning/RIT_{rit_id}_3/Algorithm/Evidence.txt')[1]
        except:
            log_L_3 = -np.inf
        max_id = np.argmax([np.median(log_L_1), np.median(log_L_2), np.median(log_L_3)])
        data = np.loadtxt(f'../src/output/nc_fits_rit_non-spinning-equal-mass/RIT_{rit_id}_{max_id+1}/Algorithm/posterior.dat')[:,[4,1,5,2,3]]
        c2A[i], c3A[i], c2p[i], c3p[i], c4p[i] = np.median(data, axis=0)
        c2A_err[i], c3A_err[i], c2p_err[i], c3p_err[i], c4p_err[i] = np.std(data, axis=0)
        full_data.append(data)
    except:
        continue

In [ ]:
colors = [matplotlib.colormaps.get_cmap('plasma')(i) for i in np.linspace(0, 1, len(rit_ids))]
ylabels = ['$c_2^A$', '$c_3^A$', '$c_2^{\\phi}$', '$c_3^{\\phi}$', '$c_4^{\\phi}$']
fig = corner.corner(full_data[0],
                    labels=ylabels,
                    color=colors[0],
                    smooth=1,
                    hist_kwargs={'histtype': 'step', 'alpha': 0.5},
                    data_kwargs={'color': colors[0],'alpha': 0.5},
                    contour_kwargs={'alpha': 0.5},
                    contourf_kwargs={'alpha': 0.5})
for i, data in enumerate(full_data):
    if i == 0:
        continue
    fig = corner.corner(data,
                        labels=ylabels,
                        color=colors[i],
                        smooth=1,
                        hist_kwargs={'histtype': 'step', 'alpha': 0.5},
                        data_kwargs={'color': colors[i],'alpha': 0.5},
                        contour_kwargs={'alpha': 0.5},
                        contourf_kwargs={'alpha': 0.5},
                        fig=fig)
fig.set_size_inches(20, 20)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(30, 6))
for i, (c, c_err, ylabel) in enumerate(zip([c2A, c3A, c2p, c3p, c4p], [c2A_err, c3A_err, c2p_err, c3p_err, c4p_err], ylabels)):
    ax[i].errorbar(ecc, c, yerr=c_err, fmt='o', color='b', markersize=5)
    ax[i].set_ylabel(ylabel)
    ax[i].set_xlabel('$e_0$')
    ax[i].set_title(ylabel)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(30, 6))
for i, (c, c_err, ylabel) in enumerate(zip([c2A, c3A, c2p, c3p, c4p], [c2A_err, c3A_err, c2p_err, c3p_err, c4p_err], ylabels)):
    ax[i].errorbar(emrg, c, yerr=c_err, fmt='o', color='b', markersize=5)
    ax[i].set_ylabel(ylabel)
    ax[i].set_xlabel('$\\hat{E}_{\\rm eff}^{\\rm mrg}$')
    ax[i].set_title(ylabel)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(30, 6))
for i, (c, c_err, ylabel) in enumerate(zip([c2A, c3A, c2p, c3p, c4p], [c2A_err, c3A_err, c2p_err, c3p_err, c4p_err], ylabels)):
    ax[i].errorbar(bmrg, c, yerr=c_err, fmt='o', color='b', markersize=5)
    ax[i].set_ylabel(ylabel)
    ax[i].set_xlabel('$\\hat{b}_{\\rm mrg}$')
    ax[i].set_title(ylabel)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(30, 6))
for i, (c, c_err, ylabel) in enumerate(zip([c2A, c3A, c2p, c3p, c4p], [c2A_err, c3A_err, c2p_err, c3p_err, c4p_err], ylabels)):
    ax[i].errorbar(jmrg, c, yerr=c_err, fmt='o', color='b', markersize=5)
    ax[i].set_ylabel(ylabel)
    ax[i].set_xlabel('$j_{\\rm mrg}$')
    ax[i].set_title(ylabel)
plt.tight_layout()
plt.show()

### Linear Fits (Single Parameter) $$Y = Y_0 + p_1 Q$$

In [ ]:
def linear_fits(fit_type, x, x_label, mask_cond, output_csv):
    y_data, y_errs = [c2A, c3A, c2p, c3p, c4p], [c2A_err, c3A_err, c2p_err, c3p_err, c4p_err]
    labels = ['$c_2^A$', '$c_3^A$', '$c_2^{\\phi}$', '$c_3^{\\phi}$', '$c_4^{\\phi}$']
    key_base = ['c_2_A', 'c_3_A', 'c_2_p', 'c_3_p', 'c_4_p']
    mask = np.where(mask_cond)
    fits, covs = [], []
    for y, e in zip(y_data, y_errs):
        p, cov = np.polyfit(x[mask], y[mask], 1, w=1/e[mask], cov=True)
        fits.append(p)
        covs.append(cov)
    for label, p in zip(labels, fits):
        print(f"{label}: {p}")

    fig, ax = plt.subplots(2, len(y_data), figsize=(30, 8), gridspec_kw={'height_ratios': [3, 1]})
    xspace = np.linspace(np.min(x[mask]), np.max(x[mask]), 100)
    for i, (top_ax, bot_ax, y, e, p, ylabel) in enumerate(zip(ax[0], ax[1], y_data, y_errs, fits, labels)):
        top_ax.plot(xspace, np.polyval(p, xspace), label='Fit', color='k', linestyle='--')
        top_ax.errorbar(x[mask], y[mask], yerr=e[mask], color='b', markersize=5, linewidth=1, capsize=3, fmt='o')
        top_ax.scatter(x[mask], y[mask], color='b')
        top_ax.set_ylabel(ylabel)
        top_ax.set_title(f'{ylabel}')
        residual = 100 * (y[mask] - np.polyval(p, x[mask])) / y[mask]
        bot_ax.axhline(0, color='k', linestyle='--')
        bot_ax.scatter(x[mask], residual, color='b')
        bot_ax.set_xlabel(f'${x_label}$')
        bot_ax.set_ylabel(f'$\\delta$ {ylabel} $[\\%]$')
    plt.tight_layout()
    plt.show()

    if output_csv:
        fit_dict = {'fit_type': fit_type, 'fit_order': int('1' * len(y_data))}
        for label, p, key in zip(labels, fits, key_base):
            fit_dict[f'{key}_0'] = p[0]
            fit_dict[f'{key}_1'] = p[1]

        pd.DataFrame([fit_dict]).to_csv(output_csv, index=False)

In [ ]:
linear_fits('ecc', ecc, 'e_0', (c2A > 0), '../src/data/fits/nc_fits_rit_non-spinning-equal-mass/order_fits_ecc.csv')

In [ ]:
linear_fits('emrg', emrg, '\\hat{E}_{\\rm eff}^{\\rm mrg}', (c2A > 0), '../src/data/fits/nc_fits_rit_non-spinning-equal-mass/order_fits_emrg.csv')

In [ ]:
linear_fits('bmrg', bmrg, '\\hat{b}_{\\rm mrg}', (c2A > 0), '../src/data/fits/nc_fits_rit_non-spinning-equal-mass/order_fits_bmrg.csv')

In [ ]:
linear_fits('jmrg', jmrg, 'j_{\\rm mrg}', (c2A > 0), '../src/data/fits/nc_fits_rit_non-spinning-equal-mass/order_fits_jmrg.csv')

### Global Mismatches (Linear Fits: Single Parameter)

In [ ]:
mismatch = np.zeros(len(rit_ids))
for i, rit_id in enumerate(rit_ids):
    try:
        try:
            log_L_1 = np.genfromtxt(f'../src/output/nc_fits_rit_non-spinning-equal-mass/RIT_{rit_id}_1/Algorithm/Evidence.txt')[1]
        except:
            log_L_1 = -np.inf
        try:
            log_L_2 = np.genfromtxt(f'../src/output/nc_fits_rit_non-spinning-equal-mass/RIT_{rit_id}_2/Algorithm/Evidence.txt')[1]
        except:
            log_L_2 = -np.inf
        try:
            log_L_3 = np.genfromtxt(f'../src/output/nc_fits_rit_non-spinning-equal-mass/RIT_{rit_id}_3/Algorithm/Evidence.txt')[1]
        except:
            log_L_3 = -np.inf
        max_id = np.argmax([np.median(log_L_1), np.median(log_L_2), np.median(log_L_3)])
        mismatch_row = np.genfromtxt(f'../src/output/nc_fits_rit_non-spinning-equal-mass/RIT_{rit_id}_{max_id+1}/Algorithm/Mismatch/Mismatch_M_60_dL_410_t_s_0.0M_wDX_0.0Hz_wSX_0.0Hz_k_0.0_satDX_1.0_satSD_1.0_NFFT_869565.txt',
                                    dtype=None, names=True, encoding='utf-8', comments='#')
        mismatch_real = mismatch_row[mismatch_row['CI'] == 50][mismatch_row[mismatch_row['CI'] == 50]['Strain_data'] == 'real']['Mismatch'][0]
        mismatch_imag = mismatch_row[mismatch_row['CI'] == 50][mismatch_row[mismatch_row['CI'] == 50]['Strain_data'] == 'imag']['Mismatch'][0]
        mismatch[i] = np.sqrt(mismatch_real**2 + mismatch_imag**2)
    except:
        continue

In [ ]:
def mismatches_single(parameter):
    mismatches_rit = []
    for i, rit_id in enumerate(rit_ids):
        try:
            mismatch_row = np.genfromtxt(f'../src/output/nc_global_fits_rit_non-spinning-equal-mass/single/nc_global_fits_{parameter}_rit_non-spinning-equal-mass/RIT_{rit_id}/Algorithm/Mismatch/Mismatch_M_60_dL_410_t_s_0.0M_wDX_0.0Hz_wSX_0.0Hz_k_0.0_satDX_1.0_satSD_1.0_NFFT_869565.txt',
                                dtype=None, names=True, encoding='utf-8', comments='#')
            mismatch_real_global = mismatch_row[mismatch_row['CI'] == 50][mismatch_row[mismatch_row['CI'] == 50]['Strain_data'] == 'real']['Mismatch'][0]
            mismatch_imag_global = mismatch_row[mismatch_row['CI'] == 50][mismatch_row[mismatch_row['CI'] == 50]['Strain_data'] == 'imag']['Mismatch'][0]
            mismatches_rit.append(np.sqrt(mismatch_real_global**2 + mismatch_imag_global**2))
        except:
            mismatches_rit.append(np.nan)
    return mismatches_rit

In [ ]:
mismatch_global_ecc = mismatches_single('ecc')
mismatch_global_jmrg = mismatches_single('jmrg')
mismatch_global_emrg = mismatches_single('emrg')
mismatch_global_bmrg = mismatches_single('bmrg')

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(24, 7))
ax[0].scatter(ecc, mismatch, color='b', label='Individual fit')
ax[0].scatter(ecc, mismatch_global_ecc, color='r', label='Global fit')
ax[0].set_xlabel('$e_0$')
ax[0].set_ylabel('$\\mathcal{M}(h^{\\rm Template}, h^{\\rm NR})$')
ax[0].set_yscale('log')
ax[0].legend()
ax[1].scatter(emrg, mismatch, color='b', label='Individual fit')
ax[1].scatter(emrg, mismatch_global_emrg, color='r', label='Global fit')
ax[1].set_xlabel('$\\hat{E}_{\\rm eff}^{\\rm mrg}$')
ax[1].set_ylabel('$\\mathcal{M}(h^{\\rm Template}, h^{\\rm NR})$')
ax[1].set_yscale('log')
ax[1].legend()
ax[2].scatter(bmrg, mismatch, color='b', label='Individual fit')
ax[2].scatter(bmrg, mismatch_global_bmrg, color='r', label='Global fit')
ax[2].set_xlabel('$\\hat{b}_{\\rm mrg}$')
ax[2].set_ylabel('$\\mathcal{M}(h^{\\rm Template}, h^{\\rm NR})$')
ax[2].set_yscale('log')
ax[2].legend()
ax[3].scatter(jmrg, mismatch, color='b', label='Individual fit')
ax[3].scatter(jmrg, mismatch_global_jmrg, color='r', label='Global fit')
ax[3].set_xlabel('$j_{{\\rm mrg}}$')
ax[3].set_ylabel('$\\mathcal{M}(h^{\\rm Template}, h^{\\rm NR})$')
ax[3].set_yscale('log')
ax[3].legend()
plt.suptitle('Mismatch $\\mathcal{M}(h^{\\rm Template}, h^{\\rm NR})=1-\\langle h^{\\rm Template}| h^{\\rm NR}\\rangle$')
plt.tight_layout()
plt.show()

### Polynomial Fits (Dual Parameters)

In [ ]:
def dual_poly_fits(order,
                   fit_type,
                   Q1, Q2,
                   Q1_label, Q2_label,
                   mask_cond,
                   output_csv=None,
                   individual_bounds=(-100, 100),
                   global_bounds=[(-9.9, 10), (-10, 0.9), (0.1, 10), (-1, 10), (-1, 10)],
                   iterations=100):

    y_data = [c2A, c3A, c2p, c3p, c4p]
    y_errs = [c2A_err, c3A_err, c2p_err, c3p_err, c4p_err]
    labels = ['$c_2^A$', '$c_3^A$', '$c_2^{\\phi}$', '$c_3^{\\phi}$', '$c_4^{\\phi}$']
    mask = np.where(mask_cond)
    fits = []

    def poly(Q1_, Q2_, current_order):
        if current_order == 1:
            return np.vstack([np.ones_like(Q1_),Q1_,Q2_]).T
        elif current_order == 2:
            return np.vstack([
                np.ones_like(Q1_),Q1_,Q2_,Q1_**2,Q2_**2,Q1_*Q2_]).T
        elif current_order == 3:
            return np.vstack([
                np.ones_like(Q1_),Q1_,Q2_,Q1_**2,Q2_**2,Q1_*Q2_,Q1_**3,Q2_**3,Q1_**2*Q2_,Q1_*Q2_**2]).T
        else:
            raise ValueError("Order must be 1, 2, or 3.")

    Q1g, Q2g = np.meshgrid(
        np.linspace(np.min(Q1), np.max(Q1), 20),
        np.linspace(np.min(Q2), np.max(Q2), 20)
    )
    Dg_orig = poly(Q1g.ravel(), Q2g.ravel(), order)

    for i, (Y, E, lbl) in enumerate(zip(y_data, y_errs, labels)):
        Y_masked, E_masked = Y[mask], E[mask]
        Q1_masked, Q2_masked = Q1[mask], Q2[mask]

        design_matrix_orig = poly(Q1_masked, Q2_masked, order)
        num_coeffs_orig = design_matrix_orig.shape[1]
        
        weights = 1.0 / E_masked
        A_weighted = design_matrix_orig * weights[:, np.newaxis]
        b_weighted = Y_masked * weights

        def objective_func(p, A_w, b_w):
            return np.sum(A_w @ p - b_w)**2 + np.sum(p**2)

        lower_global, upper_global = global_bounds[i]
        
        constraints = [
            {'type': 'ineq', 'fun': lambda p, Dg, lower: Dg @ p - lower, 'args': (Dg_orig, lower_global)},
            {'type': 'ineq', 'fun': lambda p, Dg, upper: upper - Dg @ p, 'args': (Dg_orig, upper_global)}
        ]
        coeff_bnds = [individual_bounds] * num_coeffs_orig

        p0_list = []
        mean_guess = np.mean([lower_global, upper_global])
        p0_list.append(np.full(num_coeffs_orig, mean_guess))
        
        try:
            unconstrained_sol = sp.optimize.lsq_linear(A_weighted, b_weighted, bounds=individual_bounds)
            if unconstrained_sol.success:
                p0_list.append(unconstrained_sol.x)
        except:
             print(f"Unconstrained pre-fit failed for {lbl}, skipping this guess.")

        for _ in range(iterations):
            p0_list.append(np.random.uniform(low=individual_bounds[0], high=individual_bounds[1], size=num_coeffs_orig))

        best_sol = None
        for p0 in p0_list:
            sol = sp.optimize.minimize(
                fun=objective_func, x0=p0, args=(A_weighted, b_weighted),
                method='SLSQP', bounds=coeff_bnds, constraints=constraints,
            )
            if sol.success and (best_sol is None or sol.fun < best_sol.fun):
                best_sol = sol

        fit_coeffs = None
        if best_sol is not None:
            fit_coeffs = best_sol.x
            print(f"✅ Fit for {lbl} succeeded at order {order}: {np.round(fit_coeffs, 3)}")
        else:
            print(f"🛑 Optimization failed for {lbl} at order {order}. Attempting smaller orders.")

            for smaller_order in range(order - 1, 0, -1):
                print(f"   ... Rerunning for {lbl} with order={smaller_order}")
                
                design_matrix_small = poly(Q1_masked, Q2_masked, smaller_order)
                A_weighted_small = design_matrix_small * weights[:, np.newaxis]
                num_coeffs_small = design_matrix_small.shape[1]

                Dg_small = poly(Q1g.ravel(), Q2g.ravel(), smaller_order)
                constraints_small = [
                    {'type': 'ineq', 'fun': lambda p, Dg, lower: Dg @ p - lower, 'args': (Dg_small, lower_global)},
                    {'type': 'ineq', 'fun': lambda p, Dg, upper: upper - Dg @ p, 'args': (Dg_small, upper_global)}
                ]
                coeff_bnds_small = [individual_bounds] * num_coeffs_small
                
                p0_list_small = [np.random.uniform(low=individual_bounds[0], high=individual_bounds[1], size=num_coeffs_small) for _ in range(iterations)]
                
                best_sol_small = None
                for p0_small in p0_list_small:
                    sol_small = sp.optimize.minimize(
                        fun=objective_func, x0=p0_small, args=(A_weighted_small, b_weighted),
                        method='SLSQP', bounds=coeff_bnds_small, constraints=constraints_small,
                    )
                    if sol_small.success and (best_sol_small is None or sol_small.fun < best_sol_small.fun):
                        best_sol_small = sol_small

                if best_sol_small is not None:
                    print(f"✅ Fit for {lbl} succeeded at lower order {smaller_order}.")
                    fit_coeffs = np.zeros(num_coeffs_orig)
                    successful_coeffs = best_sol_small.x
                    fit_coeffs[:len(successful_coeffs)] = successful_coeffs
                    print(f"   ... Padded coeffs: {np.round(fit_coeffs, 3)}")
                    break
            
            if fit_coeffs is None:
                print(f"🛑 Ultimate Failed for {lbl} even at smallest orders. Appending zeros.")
                fit_coeffs = np.ones(num_coeffs_orig)*np.mean([np.min(fits), np.max(fits)])

        fits.append(fit_coeffs)

    fig = plt.figure(figsize=(30, 12))
    gs = fig.add_gridspec(3, 5, height_ratios=[10, 1, 1])
    ax3d = [fig.add_subplot(gs[0, j], projection='3d') for j in range(len(y_data))]
    ax1 = [fig.add_subplot(gs[1, j]) for j in range(len(y_data))]
    ax2 = [fig.add_subplot(gs[2, j]) for j in range(len(y_data))]

    for j, (a3, a1, a2, Y, E, p, lbl) in enumerate(zip(ax3d, ax1, ax2, y_data, y_errs, fits, labels)):
        surf = Dg_orig.dot(p).reshape(Q1g.shape)
        a3.plot_surface(Q1g, Q2g, surf, alpha=0.125, color='r', rstride=1, cstride=1, edgecolor='none')
        
        a3.scatter(Q1, Q2, Y, color='b', alpha=0.125)
        a3.scatter(Q1[mask], Q2[mask], Y[mask], color='b', marker='o')
        a3.set_xlabel(Q1_label)
        a3.set_ylabel(Q2_label)
        a3.set_zlabel(lbl)
        a3.set_title(lbl, fontsize=18)

        res = 100 * (poly(Q1, Q2, order).dot(p) - Y) / Y

        for ax_res, Q_res, Q_res_label in [(a1, Q1, Q1_label), (a2, Q2, Q2_label)]:
            ax_res.axhline(0, linestyle='--', color='b')
            ax_res.scatter(Q_res[~mask_cond], res[~mask_cond], color='b', alpha=0.125)
            ax_res.scatter(Q_res[mask_cond], res[mask_cond], color='b')
            ax_res.set_xlabel(Q_res_label)
            ax_res.set_ylabel(f'$\\delta$ {lbl} $[\\%]$')
    
    plt.tight_layout()
    plt.show()

    fit_dict = {'fit_type': fit_type, 'fit_order': int('{}'.format(order)*5)}
    if output_csv:
        for lbl, p in zip(labels, fits):
            key = lbl.strip('$').replace('^{\\phi}','_p').replace('^A','_A').replace('{','_').replace('}','')
            if order == 1:
                fit_dict.update({
                f'{key}_y0':    p[0], f'{key}_p11': p[1], f'{key}_p12': p[2]
                })
            elif order == 2:
                fit_dict.update({
                f'{key}_y0':    p[0], f'{key}_p11': p[1], f'{key}_p12': p[2],
                f'{key}_p21':   p[3], f'{key}_p22': p[4], f'{key}_p212': p[5]
                })
            elif order == 3:
                fit_dict.update({
                f'{key}_y0':    p[0], f'{key}_p11': p[1], f'{key}_p12': p[2],
                f'{key}_p21':   p[3], f'{key}_p22': p[4], f'{key}_p212': p[5],
                f'{key}_p31':   p[6], f'{key}_p32': p[7],
                f'{key}_p313':  p[8], f'{key}_p323': p[9]
                })
            
        pd.DataFrame([fit_dict]).to_csv(output_csv, index=False)
        print(f"\nFit coefficients saved to {output_csv}")

#### Linear Fits $$Y = Y_0 + p_{11} Q_1 + p_{12} Q_2$$

In [ ]:
dual_poly_fits(1, 'ecc_bmrg', ecc, bmrg, '$e_0$', '$\\hat{b}_{\\rm mrg}$', (c2A > 0), '../src/data/fits/nc_fits_rit_non-spinning-equal-mass/order_fits_ecc_bmrg_1.csv')

In [ ]:
dual_poly_fits(1, 'ecc_jmrg', ecc, jmrg, '$e_0$', '$j_{\\rm mrg}$', (c2A > 0), '../src/data/fits/nc_fits_rit_non-spinning-equal-mass/order_fits_ecc_jmrg_1.csv')

In [ ]:
dual_poly_fits(1, 'bmrg_jmrg', bmrg, jmrg, '$\\hat{b}_{\\rm mrg}$', '$j_{\\rm mrg}$', (c2A > 0), '../src/data/fits/nc_fits_rit_non-spinning-equal-mass/order_fits_bmrg_jmrg_1.csv')

In [ ]:
dual_poly_fits(1, 'bmrg_emrg', bmrg, emrg, '$\\hat{b}_{\\rm mrg}$', '$\\hat{E}_{\\rm eff}^{\\rm mrg}$', (c2A > 0), '../src/data/fits/nc_fits_rit_non-spinning-equal-mass/order_fits_bmrg_emrg_1.csv')

#### Quadratic Fits $$Y = Y_0 + p_{11} Q_1 + p_{12} Q_2 + p_{21} Q_1^2 + p_{22} Q_2^2 + p_{212} Q_1 Q_2$$

In [ ]:
dual_poly_fits(2, 'ecc_bmrg', ecc, bmrg, '$e_0$', '$\\hat{b}_{\\rm mrg}$', (c2A > 0), '../src/data/fits/nc_fits_rit_non-spinning-equal-mass/order_fits_ecc_bmrg_2.csv')

In [ ]:
dual_poly_fits(2, 'ecc_jmrg', ecc, jmrg, '$e_0$', '$j_{\\rm mrg}$', (c2A > 0), '../src/data/fits/nc_fits_rit_non-spinning-equal-mass/order_fits_ecc_jmrg_2.csv')

In [ ]:
dual_poly_fits(2, 'bmrg_jmrg', bmrg, jmrg, '$\\hat{b}_{\\rm mrg}$', '$j_{\\rm mrg}$', (c2A > 0), '../src/data/fits/nc_fits_rit_non-spinning-equal-mass/order_fits_bmrg_jmrg_2.csv')

In [ ]:
dual_poly_fits(2, 'bmrg_emrg', bmrg, emrg, '$\\hat{b}_{\\rm mrg}$', '$\\hat{E}_{\\rm eff}^{\\rm mrg}$', (c2A > 0), '../src/data/fits/nc_fits_rit_non-spinning-equal-mass/order_fits_bmrg_emrg_2.csv')

#### Cubic Fits $$Y = Y_0 + p_{11} Q_1 + p_{12} Q_2 + p_{21} Q_1^2 + p_{22} Q_2^2 + p_{212} Q_1 Q_2 + p_{31} Q_1^3 + p_{32} Q_2^3 + p_{313} Q_1^2 Q_2 + p_{323} Q_1 Q_2^2$$

In [ ]:
dual_poly_fits(3, 'ecc_bmrg', ecc, bmrg, '$e_0$', '$\\hat{b}_{\\rm mrg}$', (c2A > 0), '../src/data/fits/nc_fits_rit_non-spinning-equal-mass/order_fits_ecc_bmrg_3.csv')

In [ ]:
dual_poly_fits(3, 'ecc_jmrg', ecc, jmrg, '$e_0$', '$j_{\\rm mrg}$', (c2A > 0), '../src/data/fits/nc_fits_rit_non-spinning-equal-mass/order_fits_ecc_jmrg_3.csv')

In [ ]:
dual_poly_fits(3, 'bmrg_jmrg', bmrg, jmrg, '$\\hat{b}_{\\rm mrg}$', '$j_{\\rm mrg}$', (c3A < 0), '../src/data/fits/nc_fits_rit_non-spinning-equal-mass/order_fits_bmrg_jmrg_3.csv')

In [ ]:
dual_poly_fits(3, 'bmrg_emrg', bmrg, emrg, '$\\hat{b}_{\\rm mrg}$', '$\\hat{E}_{\\rm eff}^{\\rm mrg}$', (c2A > 0), '../src/data/fits/nc_fits_rit_non-spinning-equal-mass/order_fits_bmrg_emrg_3.csv')

### Global Mismatches (Polynomial Fits: Dual Parameters)

In [ ]:
def mismatches_dual(parameter):
    mismatches_rit = []
    for order in [1,2,3]:
        mismatch_rit = []
        for i, rit_id in enumerate(rit_ids):
            try:
                mismatch_row = np.genfromtxt(f'../src/output/nc_global_fits_rit_non-spinning-equal-mass/dual/nc_global_fits_{parameter}_{order}_rit_non-spinning-equal-mass/RIT_{rit_id}/Algorithm/Mismatch/Mismatch_M_60_dL_410_t_s_0.0M_wDX_0.0Hz_wSX_0.0Hz_k_0.0_satDX_1.0_satSD_1.0_NFFT_869565.txt',
                                    dtype=None, names=True, encoding='utf-8', comments='#')
                mismatch_real_global = mismatch_row[mismatch_row['CI'] == 50][mismatch_row[mismatch_row['CI'] == 50]['Strain_data'] == 'real']['Mismatch'][0]
                mismatch_imag_global = mismatch_row[mismatch_row['CI'] == 50][mismatch_row[mismatch_row['CI'] == 50]['Strain_data'] == 'imag']['Mismatch'][0]
                mismatch_rit.append(np.sqrt(mismatch_real_global**2 + mismatch_imag_global**2))
            except:
                mismatch_rit.append(np.nan)
        mismatches_rit.append(mismatch_rit)
    return mismatches_rit

In [ ]:
mismatch_global_ecc_bmrg = mismatches_dual('ecc_bmrg')
mismatch_global_ecc_jmrg = mismatches_dual('ecc_jmrg')
mismatch_global_bmrg_jmrg = mismatches_dual('bmrg_jmrg')
mismatch_global_bmrg_emrg = mismatches_dual('bmrg_emrg')

In [ ]:
fig = plt.figure(figsize=(30, 7))
ax1 = fig.add_subplot(141, projection='3d')
ax1.scatter(ecc, bmrg, mismatch, color='b', label='Individual fit')
ax1.scatter(ecc, bmrg, mismatch_global_ecc_bmrg[0], color='r', alpha=0.6, label='Global fit (Linear)')
ax1.scatter(ecc, bmrg, mismatch_global_ecc_bmrg[1], color='m', alpha=0.6, label='Global fit (Quadratic)')
ax1.scatter(ecc, bmrg, mismatch_global_ecc_bmrg[2], color='g', alpha=0.6, label='Global fit (Cubic)')
ax1.set_xlabel('$e_0$')
ax1.set_ylabel('$\\hat{b}_{\\rm mrg}$')
ax1.set_zlabel('$\\mathcal{M}(h^{\\rm Template}, h^{\\rm NR})$')
ax1.set_zlim3d(1e-7, 5e-2)
ax1.set_zscale('log')
ax2 = fig.add_subplot(142, projection='3d')
ax2.scatter(ecc, jmrg, mismatch, color='b', label='Individual fit')
ax2.scatter(ecc, jmrg, mismatch_global_ecc_jmrg[0], color='r', alpha=0.6, label='Global fit (Linear)')
ax2.scatter(ecc, jmrg, mismatch_global_ecc_jmrg[1], color='m', alpha=0.6, label='Global fit (Quadratic)')
ax2.scatter(ecc, jmrg, mismatch_global_ecc_jmrg[2], color='g', alpha=0.6, label='Global fit (Cubic)')
ax2.set_xlabel('$e_0$')
ax2.set_ylabel('$j_{\\rm mrg}$')
ax2.set_zlabel('$\\mathcal{M}(h^{\\rm Template}, h^{\\rm NR})$')
ax2.set_zlim3d(1e-7, 5e-2)
ax2.set_zscale('log')
ax3 = fig.add_subplot(143, projection='3d')
ax3.scatter(bmrg, jmrg, mismatch, color='b', label='Individual fit')
ax3.scatter(bmrg, jmrg, mismatch_global_bmrg_jmrg[0], color='r', alpha=0.6, label='Global fit (Linear)')
ax3.scatter(bmrg, jmrg, mismatch_global_bmrg_jmrg[1], color='m', alpha=0.6, label='Global fit (Quadratic)')
ax3.scatter(bmrg, jmrg, mismatch_global_bmrg_jmrg[2], color='g', alpha=0.6, label='Global fit (Cubic)')
ax3.set_xlabel('$\\hat{b}_{\\rm mrg}$')
ax3.set_ylabel('$j_{\\rm mrg}$')
ax3.set_zlabel('$\\mathcal{M}(h^{\\rm Template}, h^{\\rm NR})$')
ax3.set_zlim3d(1e-7, 5e-2)
ax3.set_zscale('log')
ax4 = fig.add_subplot(144, projection='3d')
ax4.scatter(bmrg, emrg, mismatch, color='b', label='Individual fit')
ax4.scatter(bmrg, emrg, mismatch_global_bmrg_emrg[0], color='r', alpha=0.6, label='Global fit (Linear)')
ax4.scatter(bmrg, emrg, mismatch_global_bmrg_emrg[1], color='m', alpha=0.6, label='Global fit (Quadratic)')
ax4.scatter(bmrg, emrg, mismatch_global_bmrg_emrg[2], color='g', alpha=0.6, label='Global fit (Cubic)')
ax4.set_xlabel('$\\hat{b}_{\\rm mrg}$')
ax4.set_ylabel('$\\hat{E}_{\\rm eff}^{\\rm mrg}$')
ax4.set_zlabel('$\\mathcal{M}(h^{\\rm Template}, h^{\\rm NR})$')
ax4.set_zlim3d(1e-7, 5e-2)
ax4.set_zscale('log')
plt.suptitle('Mismatch $\\mathcal{M}(h^{\\rm Template}, h^{\\rm NR})=1-\\langle h^{\\rm Template}| h^{\\rm NR}\\rangle$')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(32, 16))
ax[0,0].scatter(ecc, mismatch, color='b', label='Individual fit')
ax[0,0].scatter(ecc, mismatch_global_ecc_bmrg[0], color='r', alpha=0.6, label='Global fit (Linear)')
ax[0,0].scatter(ecc, mismatch_global_ecc_bmrg[1], color='m', alpha=0.6, label='Global fit (Quadratic)')
ax[0,0].scatter(ecc, mismatch_global_ecc_bmrg[2], color='g', alpha=0.6, label='Global fit (Cubic)')
ax[0,0].set_xlabel('$e_0$')
ax[0,0].set_ylabel('$\\mathcal{M}(h^{\\rm Template}, h^{\\rm NR})$')
ax[0,0].set_yscale('log')
ax[1,0].scatter(bmrg, mismatch, color='b', label='Individual fit')
ax[1,0].scatter(bmrg, mismatch_global_ecc_bmrg[0], color='r', alpha=0.6, label='Global fit (Linear)')
ax[1,0].scatter(bmrg, mismatch_global_ecc_bmrg[1], color='m', alpha=0.6, label='Global fit (Quadratic)')
ax[1,0].scatter(bmrg, mismatch_global_ecc_bmrg[2], color='g', alpha=0.6, label='Global fit (Cubic)')
ax[1,0].set_xlabel('$\\hat{b}_{\\rm mrg}$')
ax[1,0].set_ylabel('$\\mathcal{M}(h^{\\rm Template}, h^{\\rm NR})$')
ax[1,0].set_yscale('log')
ax[0,1].scatter(ecc, mismatch, color='b', label='Individual fit')
ax[0,1].scatter(ecc, mismatch_global_ecc_jmrg[0], color='r', alpha=0.6, label='Global fit (Linear)')
ax[0,1].scatter(ecc, mismatch_global_ecc_jmrg[1], color='m', alpha=0.6, label='Global fit (Quadratic)')
ax[0,1].scatter(ecc, mismatch_global_ecc_jmrg[2], color='g', alpha=0.6, label='Global fit (Cubic)')
ax[0,1].set_xlabel('$e_0$')
ax[0,1].set_yscale('log')
ax[1,1].scatter(jmrg, mismatch, color='b', label='Individual fit')
ax[1,1].scatter(jmrg, mismatch_global_ecc_jmrg[0], color='r', alpha=0.6, label='Global fit (Linear)')
ax[1,1].scatter(jmrg, mismatch_global_ecc_jmrg[1], color='m', alpha=0.6, label='Global fit (Quadratic)')
ax[1,1].scatter(jmrg, mismatch_global_ecc_jmrg[2], color='g', alpha=0.6, label='Global fit (Cubic)')
ax[1,1].set_xlabel('$j_{\\rm mrg}$')
ax[1,1].set_yscale('log')
ax[0,2].scatter(bmrg, mismatch, color='b', label='Individual fit')
ax[0,2].scatter(bmrg, mismatch_global_bmrg_jmrg[0], color='r', alpha=0.6, label='Global fit (Linear)')
ax[0,2].scatter(bmrg, mismatch_global_bmrg_jmrg[1], color='m', alpha=0.6, label='Global fit (Quadratic)')
ax[0,2].scatter(bmrg, mismatch_global_bmrg_jmrg[2], color='g', alpha=0.6, label='Global fit (Cubic)')
ax[0,2].set_xlabel('$\\hat{b}_{\\rm mrg}$')
ax[0,2].set_yscale('log')
ax[1,2].scatter(jmrg, mismatch, color='b', label='Individual fit')
ax[1,2].scatter(jmrg, mismatch_global_bmrg_jmrg[0], color='r', alpha=0.6, label='Global fit (Linear)')
ax[1,2].scatter(jmrg, mismatch_global_bmrg_jmrg[1], color='m', alpha=0.6, label='Global fit (Quadratic)')
ax[1,2].scatter(jmrg, mismatch_global_bmrg_jmrg[2], color='g', alpha=0.6, label='Global fit (Cubic)')
ax[1,2].set_xlabel('$j_{\\rm mrg}$')
ax[1,2].set_yscale('log')
ax[0,3].scatter(bmrg, mismatch, color='b', label='Individual fit')
ax[0,3].scatter(bmrg, mismatch_global_bmrg_emrg[0], color='r', alpha=0.6, label='Global fit (Linear)')
ax[0,3].scatter(bmrg, mismatch_global_bmrg_emrg[1], color='m', alpha=0.6, label='Global fit (Quadratic)')
ax[0,3].scatter(bmrg, mismatch_global_bmrg_emrg[2], color='g', alpha=0.6, label='Global fit (Cubic)')
ax[0,3].set_xlabel('$\\hat{b}_{\\rm mrg}$')
ax[0,3].set_yscale('log')
ax[1,3].scatter(emrg, mismatch, color='b', label='Individual fit')
ax[1,3].scatter(emrg, mismatch_global_bmrg_emrg[0], color='r', alpha=0.6, label='Global fit (Linear)')
ax[1,3].scatter(emrg, mismatch_global_bmrg_emrg[1], color='m', alpha=0.6, label='Global fit (Quadratic)')
ax[1,3].scatter(emrg, mismatch_global_bmrg_emrg[2], color='g', alpha=0.6, label='Global fit (Cubic)')
ax[1,3].set_xlabel('$\\hat{E}_{\\rm eff}^{\\rm mrg}$')
ax[1,3].set_yscale('log')
handles, labels = ax[0,0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=4, fontsize=20)
plt.show()